In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Cancer**</span>

In [3]:
study = "Cancer"

### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

---

# **Extract <span style="color:blue;">Cancer**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Filter patients (pathology confirmed)
### (OUTPUT) candidate_cohort

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Control: 
    * If pathology exists --> exclude

### <span style="color:#FF6347;">**READ**</span> file (EHR: enteredit_findings, pathology)

In [5]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

In [6]:
fn = file_name[0]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [7]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2
1,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-08-19,1
2,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2019-08-19,1


In [8]:
fn = file_name[1]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [9]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330018595,487690,2020-06-05,Malignant,R
1,4330018595,475789,2020-07-28,Malignant,R
2,4330029102,477464,2021-05-10,Benign,R


In [10]:
fn = file_name[3]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
patient_data_ie = pd.read_excel(file_path)

In [11]:
patient_data_ie.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4330018595,63027507,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2019-07-26,2019-07-08,Mammography,Y,2021-01-05,90728533,9654707
1,4330018595,63737104,Magee Womens Imaging North,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,2019-08-19,2019-08-05,Mammography,N,2021-01-05,99424137,9654707
2,4330018595,60103700,Magee Womens Imaging North,BIDIADRTTO,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,2020-06-02,2020-03-26,Mammography,N,2021-01-05,83422682,9654707


### **0. Merge BI-RADS with patient_data_ie**

In [12]:
# Merge EXAM_NAME into birads from patient_data_ie
exam_name = (
    patient_data_ie[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_NAME']]
    .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])
)

#### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [13]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [14]:
column_to_check = 'EXAM_NAME'
type_column = 'Study'

exam_name[column_to_check] = exam_name[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = exam_name[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    exam_name.loc[mask, type_column] = label

#### <span style="color:blue">**Series**</span> (DBT)

In [15]:
series_types = {
    "DBT":    ["Breast Tomosynthesis", "TOMOSYNTHESIS", "TOMOSYN", "TOMO"]
}

In [16]:
column_to_check = 'EXAM_NAME'
type_column = 'Series'

exam_name[column_to_check] = exam_name[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = exam_name[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    exam_name.loc[mask, type_column] = label

In [17]:
exam_name.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_NAME,Study,Series
0,4330018595,63027507,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
1,4330018595,63737104,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT
2,4330018595,60103700,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT
3,4330018595,60695725,BREAST ULTRASOUND LIMITED RIGHT,NaN,NaN
4,4330018595,60690108,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT


In [18]:
birads = birads.merge(exam_name, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='left')

In [19]:
birads.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series
0,4330018595,63027507,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
1,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-08-19,1,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT
2,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2019-08-19,1,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT
3,4330018595,60690108,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-06-02,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
4,4330018595,60103700,Heterogeneously dense (51% - 75%),NaN,"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2020-06-02,2,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT


### **1.Merge patients with pathology**

In [20]:
birads.drop(columns="FINDING_LOCATION", inplace=True)

In [21]:
biopsy_categories = [
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken'
]

In [22]:
# ── STEP 1: Merge birads + pathology ────────────────────────────────────────

birads['EXAM_COMPLETED_DATE'] = pd.to_datetime(birads['EXAM_COMPLETED_DATE'])
pathology['PATHOLOGY_DATE']   = pd.to_datetime(pathology['PATHOLOGY_DATE'])
birads['needs_biopsy']        = birads['FINDING_CATEGORY'].isin(biopsy_categories)

# STEP 1.1: Biopsy findings → link to pathology via date window
birads_biopsy    = birads[birads['needs_biopsy']].copy()
biopsy_with_path = pd.merge(birads_biopsy, pathology, on='PATIENT_STUDY_ID', how='left')
biopsy_with_path = biopsy_with_path[
    (biopsy_with_path['PATHOLOGY_DATE'] >= biopsy_with_path['EXAM_COMPLETED_DATE']) &
    (biopsy_with_path['PATHOLOGY_DATE'] <= biopsy_with_path['EXAM_COMPLETED_DATE'] + pd.DateOffset(months=6))
]
biopsy_with_path['days_diff'] = (
    biopsy_with_path['PATHOLOGY_DATE'] - biopsy_with_path['EXAM_COMPLETED_DATE']
).dt.days
biopsy_with_path = (
    biopsy_with_path
    .sort_values('days_diff')
    .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first')
    .drop(columns='days_diff')
)

# STEP 1.2: Non-biopsy findings → assign opposite side,
#           but ONLY when exactly one biopsy side exists (L+R = ambiguous → skip)
birads_no_biopsy = birads[~birads['needs_biopsy']].copy()

biopsy_sides = (
    biopsy_with_path[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
    .drop_duplicates()
    .rename(columns={'SIDE': 'biopsy_side'})
)
single_side_accessions = (
    biopsy_sides
    .groupby(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])
    .filter(lambda g: len(g) == 1)
)

birads_no_biopsy = pd.merge(
    birads_no_biopsy, single_side_accessions,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='left'
)
birads_no_biopsy['SIDE'] = birads_no_biopsy['biopsy_side'].map({'L': 'R', 'R': 'L'})
birads_no_biopsy = birads_no_biopsy.drop(columns='biopsy_side')

# STEP 1.3: Rows still without SIDE, accession has NO biopsy at all,
#           and duplicate_count >= 2 → bilateral exam → duplicate as L and R
biopsy_accession_keys = biopsy_sides[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()

no_side_rows  = birads_no_biopsy[birads_no_biopsy['SIDE'].isna()].copy()
has_side_rows = birads_no_biopsy[birads_no_biopsy['SIDE'].notna()].copy()

no_side_rows = no_side_rows.merge(
    biopsy_accession_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
)
truly_no_biopsy  = no_side_rows['_merge'] == 'left_only'
no_side_rows     = no_side_rows.drop(columns='_merge')

bilateral_mask = truly_no_biopsy & (no_side_rows['duplicate_count'] >= 2)
bilateral      = no_side_rows[bilateral_mask]
rest           = no_side_rows[~bilateral_mask]

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_no_biopsy = pd.concat(
    [has_side_rows, bilateral_L, bilateral_R, rest], ignore_index=True
)

birads_with_path = pd.concat([biopsy_with_path, birads_no_biopsy], ignore_index=True)

In [23]:
birads_with_path.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,needs_biopsy,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4333061886,86465954,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2015-12-16,2,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,60079.0,2015-12-16,Malignant,L
1,4333016625,89454031,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2015-02-24,1,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,79677.0,2015-02-24,Malignant,R
2,4333171887,71004907,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-06-26,1,DIAG MAMMO PROD DIRECT DIGITAL IMAGES BILATERAL,DIAG,NaN,True,441971.0,2017-06-26,Malignant,L
3,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477137.0,2021-04-28,Malignant,R
4,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477138.0,2021-04-28,Malignant,L


In [24]:
df_step1 = birads_with_path.copy()

In [25]:
df_step1["PATIENT_STUDY_ID"].nunique()

9266

### <span style="color:#FF6347;">**SAVE**</span> file

In [26]:
# # output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
# df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**Cancer**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) candidate_cohort

### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [27]:
# # output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
# control_cohort = pd.read_excel(output_file)

In [28]:
# control_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(control_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

## 1. Locate <span style="color:blue;">**Cancer**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [29]:
cohort = df_step1.copy()
cohort['Index'] = None

In [30]:
# 1. Find the earliest Malignant PATHOLOGY_DATE per patient
first_malignant = (
    cohort[cohort['LESION_CLASS'] == 'Malignant']
    .sort_values('PATHOLOGY_DATE')
    .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
    [['PATIENT_STUDY_ID', 'PATHOLOGY_DATE']]
    .rename(columns={'PATHOLOGY_DATE': 'target pathology date'})
)

cohort = cohort.merge(first_malignant, on='PATIENT_STUDY_ID', how='left')

# 2. Mark exams within 6 months BEFORE (or on) the index pathology date
six_months = 365 / 12 * 6

cohort['path to exam diff'] = (cohort['target pathology date'] - cohort['EXAM_COMPLETED_DATE']).dt.days

mask_index_cohort = (
    (cohort['path to exam diff'] >= 0) &
    (cohort['path to exam diff'] <= six_months)
)

cohort.loc[mask_index_cohort, 'Index'] = 'INDEX'
cohort.loc[~mask_index_cohort | (cohort['LESION_CLASS'] != 'Malignant'), 'path to exam diff'] = None

print(f"✅ Marked {cohort['Index'].eq('INDEX').sum()} rows as 'INDEX'.")

✅ Marked 5578 rows as 'INDEX'.


In [31]:
cohort.reset_index(drop=True, inplace=True)

In [32]:
cancer_patients = set(cohort.loc[
    (cohort['LESION_CLASS'] == 'Malignant'),
    'PATIENT_STUDY_ID'
].unique())

cancer = cohort[cohort['PATIENT_STUDY_ID'].isin(cancer_patients)].copy()
print(f"# of cancer patients: {len(cancer_patients)}")
print(f"cancer rows      : {len(cancer)}")
print(f"Index breakdown:\n{cancer['Index'].value_counts(dropna=False)}")

# of cancer patients: 1831
cancer rows      : 28150
Index breakdown:
Index
None     22572
INDEX     5578
Name: count, dtype: int64


## 2. Locate <span style="color:blue;">**Cancer**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>. Adjust <span style="color:pink;"> **screening interval (OR search period)**</span> 

Add threshold labeling for the same index time point (threshold ~1.5 months)

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [33]:
cohort = cancer.copy()

In [34]:
# ── Parameters ────────────────────────────────────────────────────────────────
min_months      = pd.Timedelta(days=0)              # how far back to look (min)
max_months      = pd.Timedelta(days=365/12 * 36)   # how far back to look (max)
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [35]:
# 1. Get the earliest INDEX exam date per patient
earliest_index = (
    cohort[cohort['Index'] == 'INDEX']
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .min()
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'earliest index date'})
)

cohort = cohort.merge(earliest_index, on='PATIENT_STUDY_ID', how='left')

# 2. Time between each exam and the earliest INDEX exam
cohort['exam to index diff'] = cohort['earliest index date'] - cohort['EXAM_COMPLETED_DATE']

# 3. Window: at least 9 months before, no more than 18 months before
mask_window = (
    (cohort['exam to index diff'] > min_months) &
    (cohort['exam to index diff'] <= max_months) &
    (cohort['Study'] == 'SCREEN')
)

cohort.loc[mask_window, 'Index'] = 'INDEX-1'


# # ── Re-rank: latest exam → INDEX-1, second latest → INDEX-2, etc. ────────────
# # Get unique exam dates per patient within the window
# date_ranks = (
#     cohort[cohort['Index'] == 'INDEX-1']
#     .drop_duplicates(subset=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'])
#     [['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE']]
#     .sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, False])
# )

# # Rank within each patient: 1 = latest (closest to INDEX), 2 = second latest, ...
# date_ranks['index_rank'] = (
#     date_ranks.groupby('PATIENT_STUDY_ID').cumcount() + 1
# )

# # Merge rank back and overwrite the Index label
# cohort = cohort.merge(
#     date_ranks, on=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], how='left'
# )
# ranked_mask = cohort['index_rank'].notna()
# cohort.loc[ranked_mask, 'Index'] = (
#     'INDEX-' + cohort.loc[ranked_mask, 'index_rank'].astype(int).astype(str)
# )
# cohort = cohort.drop(columns='index_rank')




def assign_index_labels(dates):
    dates_sorted = sorted(set(dates), reverse=True)
    labels = {}
    rank = 0
    group_anchor = None
    for d in dates_sorted:
        if group_anchor is None or (group_anchor - d) > threshold_group:
            rank += 1
            group_anchor = d
        labels[d] = f'INDEX-{rank}'
    return labels

window_rows = cohort[mask_window].copy()
date_label_map = {}
for pid, grp in window_rows.groupby('PATIENT_STUDY_ID'):
    mapping = assign_index_labels(grp['EXAM_COMPLETED_DATE'])
    for date, label in mapping.items():
        date_label_map[(pid, date)] = label

cohort.loc[mask_window, 'Index'] = cohort.loc[mask_window].apply(
    lambda row: date_label_map[(row['PATIENT_STUDY_ID'], row['EXAM_COMPLETED_DATE'])],
    axis=1
)


print(f"✅ Marked {cohort['Index'].eq('INDEX-1').sum()} rows as 'INDEX-1'.")
print(f"   Patients with INDEX-1 : {cohort[cohort['Index'] == 'INDEX-1']['PATIENT_STUDY_ID'].nunique()}")
print(f"   Patients with INDEX   : {cohort[cohort['Index'] == 'INDEX']['PATIENT_STUDY_ID'].nunique()}")

✅ Marked 2082 rows as 'INDEX-1'.
   Patients with INDEX-1 : 1046
   Patients with INDEX   : 1831


In [36]:
cohort = cohort.drop(columns = ["target pathology date", "earliest index date", "exam to index diff"])

In [37]:
cohort.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,needs_biopsy,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,Index,path to exam diff
0,4333061886,86465954,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2015-12-16,2,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,60079.0,2015-12-16,Malignant,L,INDEX,0.0
1,4333016625,89454031,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2015-02-24,1,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,79677.0,2015-02-24,Malignant,R,INDEX,0.0
2,4333171887,71004907,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-06-26,1,DIAG MAMMO PROD DIRECT DIGITAL IMAGES BILATERAL,DIAG,NaN,True,441971.0,2017-06-26,Malignant,L,INDEX,0.0
3,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477137.0,2021-04-28,Malignant,R,INDEX,0.0
4,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477138.0,2021-04-28,Malignant,L,INDEX,0.0


In [38]:
# ── Time-to-INDEX column ──────────────────────────────────────────────────────
# Use the earliest INDEX exam date per patient (the reference anchor)
index_dates = (
    cohort[cohort['Index'] == 'INDEX']
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .min()
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'INDEX_DATE'})
)

cohort = cohort.merge(index_dates, on='PATIENT_STUDY_ID', how='left')

is_prior = cohort['Index'].str.match(r'^INDEX-\d+$', na=False)

# Compute days directly — avoids the object-dtype problem
cohort['time_to_INDEX_days'] = (
    cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']
).dt.days

# Zero out rows that aren't INDEX-N (set to NaN instead of 0)
cohort.loc[~is_prior, 'time_to_INDEX_days'] = pd.NA

cohort['time_to_INDEX_months'] = (cohort['time_to_INDEX_days'] / 30.5).round(1)
cohort = cohort.drop(columns='INDEX_DATE')

cohort['path_to_INDEX_months'] = (cohort['path to exam diff'] / 30.5).round(1)
cohort = cohort.drop(columns='path to exam diff')

# print(cohort[is_prior][['PATIENT_STUDY_ID','Index','EXAM_COMPLETED_DATE',
#                           'time_to_INDEX_days','time_to_INDEX_months']].head(10))

### <span style="color:#FF6347;">**SAVE**</span> file

In [39]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'candidate_cohort' + ".xlsx")
# cohort.to_excel(output_file, index=False)

In [40]:
cohort["PATIENT_STUDY_ID"].nunique()

1831

In [41]:
df_step2 = cohort.copy()

In [42]:
df_step2.drop(columns=["needs_biopsy", "duplicate_count"], inplace=True)

In [43]:
df_step2.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,EXAM_NAME,Study,Series,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months,path_to_INDEX_months
0,4333061886,86465954,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2015-12-16,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,60079.0,2015-12-16,Malignant,L,INDEX,NaN,NaN,0.0
1,4333016625,89454031,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2015-02-24,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,79677.0,2015-02-24,Malignant,R,INDEX,NaN,NaN,0.0
2,4333171887,71004907,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-06-26,DIAG MAMMO PROD DIRECT DIGITAL IMAGES BILATERAL,DIAG,NaN,441971.0,2017-06-26,Malignant,L,INDEX,NaN,NaN,0.0
3,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,477137.0,2021-04-28,Malignant,R,INDEX,NaN,NaN,0.0
4,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,477138.0,2021-04-28,Malignant,L,INDEX,NaN,NaN,0.0


In [44]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")
cohort["PATHOLOGY_DATE"] = pd.to_datetime(cohort["PATHOLOGY_DATE"], format="%Y-%m-%d")

In [45]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_candidate_cohort' + ".xlsx")
cohort.to_excel(output_file, index=False)

## **STEP 3**. Extract <span style="color:blue;">**MO cancer**</span> cohort only
### (Output) mo_cancer_candidate, mo_cancer_candidate_cohort

#### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [46]:
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_candidate_cohort' + ".xlsx")
cancer_cohort = pd.read_excel(output_file)

In [47]:
cancer_cohort.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,needs_biopsy,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months,path_to_INDEX_months
0,4333061886,86465954,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2015-12-16,2,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,60079.0,2015-12-16,Malignant,L,INDEX,NaN,NaN,0.0
1,4333016625,89454031,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2015-02-24,1,DIAGNOSTIC MAMMOGRAPHY PRODUCING DIRECT DIGITA...,DIAG,NaN,True,79677.0,2015-02-24,Malignant,R,INDEX,NaN,NaN,0.0
2,4333171887,71004907,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-06-26,1,DIAG MAMMO PROD DIRECT DIGITAL IMAGES BILATERAL,DIAG,NaN,True,441971.0,2017-06-26,Malignant,L,INDEX,NaN,NaN,0.0
3,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477137.0,2021-04-28,Malignant,R,INDEX,NaN,NaN,0.0
4,4333807317,66017863,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2021-04-28,1,BRST IMG CONTRAST ENHANCED TOMO BI RESEARCH ON...,NaN,DBT,True,477138.0,2021-04-28,Malignant,L,INDEX,NaN,NaN,0.0


### **Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "LESION_CLASS" = "Malignant"
##### no need to check as all patients in cancer_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN"

In [48]:
normal_cats = ['1 - Negative', '2 - Benign finding']

recall_cats = [
    '0 - Need additional imaging evaluation',
    '3 - Probably benign - short interval follow-up',
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken',
    '6 - Known biopsy proven malignancy'
]

In [49]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cancer_cohort.loc[
    (cancer_cohort['Index'] == 'INDEX-1') &
    (cancer_cohort['Series'] == 'DBT') &
    (cancer_cohort['FINDING_CATEGORY'].isin(recall_cats)) &
    (cancer_cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients = set(cancer_cohort.loc[
    (cancer_cohort['Index'] == 'INDEX-1') &
    (cancer_cohort['Series'] == 'DBT') &
    (cancer_cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cancer_cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

mo_cancer = cancer_cohort[cancer_cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"mo_cancer rows      : {len(mo_cancer)}")
print(f"Index breakdown:\n{mo_cancer['Index'].value_counts(dropna=False)}")

Normal patients     : 803
Recall patients     : 180
Overlap removed     : 150
Qualifying patients : 653
mo_cancer rows      : 10553
Index breakdown:
Index
NaN        6174
INDEX      2128
INDEX-1    1308
INDEX-2     889
INDEX-3      54
Name: count, dtype: int64


In [50]:
mo_cancer_cohort = cancer_cohort[cancer_cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
mo_cancer_cohort = mo_cancer_cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE']).reset_index(drop=True)

mo_cancer_cohort["EXAM_COMPLETED_DATE"] = mo_cancer_cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
mo_cancer_cohort["PATHOLOGY_DATE"] = mo_cancer_cohort["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

cancer_cohort["EXAM_COMPLETED_DATE"] = cancer_cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
cancer_cohort["PATHOLOGY_DATE"] = cancer_cohort["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_cohort**

In [51]:
mo_cancer_cohort['PATIENT_STUDY_ID'].nunique()

653

In [52]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'mo_cancer_candidate_cohort' + ".xlsx")

mo_cancer_cohort.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer** (only retain INDX & INDEX-1)

In [53]:
mo_cancer= mo_cancer_cohort[(mo_cancer_cohort["Index"]=="INDEX") | (mo_cancer_cohort["Index"]=="INDEX-1")]

In [54]:
mo_cancer['PATIENT_STUDY_ID'].nunique()

653

In [55]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'mo_cancer_candidate' + ".xlsx")

mo_cancer.to_excel(output_file, index=False)

In [62]:
mo_cancer[(mo_cancer["Index"]=="INDEX-1")&(mo_cancer["Series"]=="DBT")]["PATIENT_STUDY_ID"].nunique(), mo_cancer[(mo_cancer["Index"]=="INDEX")&(mo_cancer["Series"]=="DBT")]["PATIENT_STUDY_ID"].nunique()

(653, 648)

##### <span style="color:#FF6347;">**SAVE**</span> **cancer_candidate_cohort** ("DBT" available @ "INDEX")

In [56]:
pid_cancer_w_dbt_index = cancer_cohort[(cancer_cohort["Index"]=="INDEX")&(cancer_cohort["Series"]=="DBT")]["PATIENT_STUDY_ID"].unique()
cancer_cohort_dbt =  cancer_cohort[cancer_cohort["PATIENT_STUDY_ID"].isin(pid_cancer_w_dbt_index)]

In [57]:
len(pid_cancer_w_dbt_index), cancer_cohort_dbt["PATIENT_STUDY_ID"].nunique()

(1630, 1630)

In [58]:
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_candidate_cohort' + ".xlsx")

cancer_cohort_dbt.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **cancer** ("DBT" available @ "INDEX") (only retain INDEX & INDEX-1)

In [59]:
cancer_dbt = cancer_cohort_dbt[(cancer_cohort_dbt["Index"]=="INDEX") | (cancer_cohort_dbt["Index"]=="INDEX-1")]

In [60]:
cancer_dbt['PATIENT_STUDY_ID'].nunique()

1630

In [61]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_candidate' + ".xlsx")

cancer_dbt.to_excel(output_file, index=False)

In [63]:
pids_cancer_dbt_index = cancer_cohort_dbt[(cancer_cohort_dbt["Index"]=="INDEX")&(cancer_cohort_dbt["Series"]=="DBT")]["PATIENT_STUDY_ID"].unique()
cancer_dbt_index = cancer_cohort_dbt[cancer_cohort_dbt["PATIENT_STUDY_ID"].isin(pids_cancer_dbt_index)]

In [64]:
cancer_dbt_index[(cancer_dbt_index["Index"]=="INDEX")&(cancer_dbt_index["Series"]=="DBT")]["PATIENT_STUDY_ID"].nunique(), cancer_dbt_index[(cancer_dbt_index["Index"]=="INDEX-1")&(cancer_dbt_index["Series"]=="DBT")]["PATIENT_STUDY_ID"].nunique()

(1630, 821)

In [65]:
pids_cancer_dbt_index_1 = cancer_cohort[(cancer_cohort["Index"]=="INDEX-1")&(cancer_cohort["Series"]=="DBT")]["PATIENT_STUDY_ID"].unique()
cancer_dbt_index_1 = cancer_cohort[cancer_cohort["PATIENT_STUDY_ID"].isin(pids_cancer_dbt_index_1)]

In [66]:
cancer_dbt_index_1["PATIENT_STUDY_ID"].nunique(), cancer_dbt_index_1[(cancer_dbt_index_1["Index"]=="INDEX")&(cancer_dbt_index_1["Series"]=="DBT")]["PATIENT_STUDY_ID"].nunique()

(833, 821)